# IAC-Flow — ToothFairy3 Left/Right IAC — Colab uçtan uca çalıştırıcı (v1.1)

Bu notebook **README'deki 8 adımlık boru hattının tamamını** sırayla çalıştırır.
Yukarıdan aşağıya, hücreleri sırayla çalıştır. Her hücre bir repo Python dosyasını çağırır;
hiçbir mantık burada tekrar yazılmadı, sadece doğru sırayla tetikleniyor.

## Önce 3 şeyi bil (düz anlatım)

1. **Kod repoda, notebook sadece 'çağırıcı'.** `data/`, `nnunet/`, `flow/` içindeki `.py`
   dosyaları işin kendisi. Eski notebook bu dosyaların bazılarını çağırmıyordu (fold üretme,
   OOF tahmin, SDF cache eksikti). Bu sürüm hepsini sırayla çağırır. Yani repoda dosya var
   çünkü **iş orada**; notebook onları çalıştıran bir düğme paneli.

2. **Eğitim için GPU (Colab) şart.** CPU sadece `selftest.py` ve `tests/` için yeterli
   (mekanizmayı saniyeler içinde doğrular). Gerçek nnU-Net ve flow eğitimi GPU ister.
   Menü: **Runtime → Change runtime type → GPU**.

3. **Evet, saatler–günler sürer.** Kaba tablo (T4 için):

   | Adım | Kısa (50 epoch) | Tam (varsayılan) |
   |---|---|---|
   | nnU-Net 1 fold | ~1–3 saat | ~1–2 gün |
   | nnU-Net 5 fold (OOF için ŞART) | ~½–1 gün | birkaç gün |
   | SDF cache (480 hacim) | ~20–40 dk | aynı |
   | Flow 1 fold | ~3–8 saat | ~½–1 gün |

   Colab oturumu kopabilir; bu yüzden checkpoint'ler Drive'a yazılır ve **her şey
   devam ettirilebilir** (nnU-Net `--c`, flow `--resume`). Aynı hücreyi tekrar
   çalıştırınca kaldığı yerden devam eder.

> **İlk denemede `FAST = True` bırak** (aşağıda). 50-epoch trainer + hızlı ayarlarla
> tüm zincirin uçtan uca çalıştığını 1 günde görürsün. Zincir çalışıyorsa `FAST = False`
> yapıp yayın-kalitesi tam eğitimi başlatırsın.

## 0. GPU kontrol + Drive bağla + repo kodunu getir
Kod her oturumda runtime'a gelmeli (Drive'a değil). GitHub'dan klonla **veya** repo
klasörünü Drive'a atıp oradan kopyala. Aşağıda ikisi de var — birini seç.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — Runtime>GPU sec!")
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# --- Repo kodunu runtime'a getir ---
# Secenek A: GitHub'dan klonla (repo public/erisilebilir ise en temizi)
REPO_URL = "https://github.com/<KULLANICI>/ToothFairy3-IAC-Segmentation-Flow.git"  # <-- DUZENLE
REPO = "/content/ToothFairy3-IAC-Segmentation-Flow"
if not os.path.isdir(REPO):
    # A) klon:
    os.system(f"git clone {REPO_URL} {REPO}")
    # B) VEYA repoyu Drive'a attiysan bunu kullan (ustteki klon satirini yorumla):
    # import shutil; shutil.copytree('/content/drive/MyDrive/ToothFairy3-IAC-Segmentation-Flow', REPO)
assert os.path.isdir(REPO), f"repo yok: {REPO} — REPO_URL'i duzelt veya Drive kopyasini kullan"
os.chdir(REPO)
print("cwd:", os.getcwd())
print("icerik:", sorted(os.listdir(REPO))[:12])

## 0.5 Dataset'i bul (ToothFairy3)

Aşağıdaki hücre dataset'i şu öncelikle bulur:
1. **Drive'da zaten açık klasör** (senin durumun, zipsiz) → `DRIVE_DIR`. Drive'dan doğrudan
   okunur, gereksiz kopya yapılmaz.
2. Daha önce runtime'a açılmış kopya.
3. Sadece `ToothFairy3.zip` varsa onu açar.

`DRIVE_DIR`'i Drive'daki gerçek klasörüne göre düzelt (içinde `imagesTr/`, `labelsTr/`,
`dataset.json` olmalı). Bulduğu yolu bir sonraki hücre `TF3_SRC` olarak otomatik alır.

> Not: ToothFairy3 login arkasındadır (otomatik indirilemez); veriyi sen sağladın, biz sadece kullanıyoruz. Kullanımda dataset makalelerine atıf gerekir.

In [ ]:
import os, zipfile, glob
# Dataset'i bul. Oncelik: (1) Drive'da zaten ACIK klasor  (2) runtime'a acilmis  (3) zip'ten ac.
DRIVE_DIR  = "/content/drive/MyDrive/ToothFairy3"       # <-- Drive'daki ACIK klasorun (zipsiz)
ZIP_PATH   = "/content/drive/MyDrive/ToothFairy3.zip"   # (yalnizca zip halinde varsa)
EXTRACT_TO = "/content/work/tf3_src"                    # zip acilirsa buraya

def find_tf3_root(base):
    """imagesTr + labelsTr + dataset.json iceren klasoru bul (ic ice olabilir)."""
    if not base or not os.path.isdir(base):
        return None
    # once direkt base'e bak, sonra alt klasorleri tara
    cand = [base] + [os.path.dirname(dj) for dj in
                     glob.glob(os.path.join(base, "**", "dataset.json"), recursive=True)]
    for root in cand:
        if (os.path.isdir(os.path.join(root, "imagesTr"))
                and os.path.isdir(os.path.join(root, "labelsTr"))
                and os.path.isfile(os.path.join(root, "dataset.json"))):
            return root
    return None

# (1) Drive'daki acik klasor  (SENIN DURUMUN: zipsiz)  -> Drive'dan direkt okunur, kopya YOK
root = find_tf3_root(DRIVE_DIR)
if root:
    print("Drive'daki acik klasor kullaniliyor (kopyalanmadan):", root)
else:
    # (2) daha once runtime'a acildiysa
    root = find_tf3_root(EXTRACT_TO)
    if root is None:
        # (3) sadece zip varsa ac
        assert os.path.isfile(ZIP_PATH), (
            f"Dataset bulunamadi.\n  Acik klasor beklenen: {DRIVE_DIR}\n  Zip beklenen: {ZIP_PATH}\n"
            "Drive'daki gercek yolu DRIVE_DIR'e yaz (icinde imagesTr/labelsTr/dataset.json olmali).")
        os.makedirs(EXTRACT_TO, exist_ok=True)
        print("zip aciliyor (birkac dk surebilir)...", flush=True)
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_TO)
        root = find_tf3_root(EXTRACT_TO)

assert root, "imagesTr/labelsTr/dataset.json bulunamadi — DRIVE_DIR'i kontrol et."
print("TF3 kok ->", root)
print("  imagesTr:", len(os.listdir(os.path.join(root, 'imagesTr'))),
      "| labelsTr:", len(os.listdir(os.path.join(root, 'labelsTr'))))
print("\n>>> Sonraki hucre TF3_SRC'yi otomatik bu yola ayarlayacak.")

import os
# 0.5 hucresi zip'i actiysa 'root' degiskeni dolu gelir; onu kullan. Yoksa Drive klasoru.
TF3_SRC = globals().get("root") or "/content/drive/MyDrive/ToothFairy3"   # <-- gerekiyorsa DUZENLE
DRIVE_RUNS = "/content/drive/MyDrive/iac_runs"    # checkpoint/sonuclar buraya (kalici)
FAST = True                                       # ilk deneme icin True; tam egitim icin False

# --- turetilen yollar ---
WORK = "/content/work"
os.environ["nnUNet_raw"]          = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = f"{DRIVE_RUNS}/nnUNet_results"   # Drive: oturum kopsa da kalir
for d in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    os.makedirs(os.environ[d], exist_ok=True)
os.makedirs(DRIVE_RUNS, exist_ok=True)

DST     = f"{os.environ['nnUNet_raw']}/Dataset801_IAC_LR"
IMAGES  = f"{DST}/imagesTr"
LABELS  = f"{DST}/labelsTr"
TRAINER = "nnUNetTrainerIAC_NoMirror_50ep" if FAST else "nnUNetTrainerIAC_NoMirror"

assert os.path.isdir(TF3_SRC), f"bulunamadi: {TF3_SRC} — TF3_SRC'yi duzelt (0.5 hucresini calistirdin mi?)"
print("TF3_SRC =", TF3_SRC)
print("kaynak OK:", sorted(os.listdir(TF3_SRC))[:5])
print("FAST =", FAST, "| trainer =", TRAINER)
print("nnUNet_results ->", os.environ["nnUNet_results"])

In [ ]:
import os
TF3_SRC = "/content/drive/MyDrive/ToothFairy3"   # <-- DUZENLE: Drive'daki ToothFairy3 klasoru
DRIVE_RUNS = "/content/drive/MyDrive/iac_runs"    # checkpoint/sonuclar buraya (kalici)
FAST = True                                       # ilk deneme icin True; tam egitim icin False

# --- turetilen yollar ---
WORK = "/content/work"
os.environ["nnUNet_raw"]          = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = f"{DRIVE_RUNS}/nnUNet_results"   # Drive: oturum kopsa da kalir
for d in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    os.makedirs(os.environ[d], exist_ok=True)
os.makedirs(DRIVE_RUNS, exist_ok=True)

DST     = f"{os.environ['nnUNet_raw']}/Dataset801_IAC_LR"
IMAGES  = f"{DST}/imagesTr"
LABELS  = f"{DST}/labelsTr"
TRAINER = "nnUNetTrainerIAC_NoMirror_50ep" if FAST else "nnUNetTrainerIAC_NoMirror"

assert os.path.isdir(TF3_SRC), f"bulunamadi: {TF3_SRC} — TF3_SRC'yi duzelt"
print("kaynak OK:", sorted(os.listdir(TF3_SRC))[:5])
print("FAST =", FAST, "| trainer =", TRAINER)
print("nnUNet_results ->", os.environ["nnUNet_results"])

## 2. Bağımlılıklar

In [ ]:
!pip -q install nnunetv2 nibabel scipy scikit-image pyyaml

## 3. (Opsiyonel ama önerilir) CPU self-test — zinciri kurmadan mekanizmayı doğrula
Saniyeler sürer, GPU/veri gerektirmez. Yeşil geçerse residual-flow makinesi sağlam.

In [ ]:
!python flow/selftest.py

---
# ADIM 1 — 3 sınıflı dataset kur (77 → {0=bg, 1=Left, 2=Right})
IAC id'leri `dataset.json`'dan **isimle** çözülür (mandibular-incisive/lingual kanalları
karışmasın diye). Bulamazsa RAISE eder — sessiz fallback yok. `--copy-images` çünkü Colab
Drive'a güvenilir symlink atamaz. `FAST` iken sadece 40 vaka işlenir (hızlı prova).

In [ ]:
LIMIT = "--limit 40" if FAST else ""
!python data/prepare_iac_dataset.py --src "$TF3_SRC" --dst "$DST" --copy-images --workers 4 $LIMIT
print("imagesTr:", len(os.listdir(IMAGES)), "| labelsTr:", len(os.listdir(LABELS)))

# ADIM 2 — split üret → `configs/splits.json`
P+F = development (5-fold CV), S = held-out external OOD test. Bu dosya hem nnU-Net OOF
hem flow eğitiminin **tek doğruluk kaynağı**. `--src` ToothFairy3 kökünü okur (etiket id'leri).

In [ ]:
!python data/create_folds.py --src "$TF3_SRC" --out configs/splits.json
import json; s = json.load(open("configs/splits.json"))
print("folds:", len(s["folds"]), "| external S:", len(s["external_test"]),
      "| dev:", len(s["development"]))

---
# TRACK A — nnU-Net v2 baseline

## A1. Plan & preprocess (dataset bütünlüğünü de doğrular)

In [ ]:
!nnUNetv2_plan_and_preprocess -d 801 --verify_dataset_integrity

## A2. Özel trainer'ları nnU-Net paketine kur (L/R için mirroring KAPALI)
Sol/sağ mirroring 1↔2 sınıflarını takas eder — domain kuralı gereği kapalı.

In [ ]:
import os, shutil, nnunetv2
_da = os.path.join(os.path.dirname(nnunetv2.__file__),
                   "training", "nnUNetTrainer", "variants", "data_augmentation")
shutil.copy("nnunet/nnUNetTrainerIAC.py", _da)
shutil.copy("nnunet/trainer_no_lr_mirror.py", _da)
print("trainer'lar kuruldu ->", _da)

## A3. 5 fold'un HEPSİNİ eğit — leakage-free OOF için ŞART
Flow'un başlangıç noktası (x0), her vakanın **onu eğitiminde görmemiş** bir modelce
tahminidir. Bunun için 5 fold da eğitilmeli. **En uzun adım budur.**

- Oturum koparsa: bu hücreyi **tekrar çalıştır** — `--c` ile kaldığı fold/epoch'tan devam eder.
- Checkpoint'ler Drive'da (`nnUNet_results`), kaybolmaz.
- Sadece 1 fold'la hızlı bakmak istersen `FOLDS = [0]` yap (ama o zaman OOF/flow tam olmaz).

In [ ]:
FOLDS = [0, 1, 2, 3, 4]     # tam OOF icin hepsi; hizli bakis icin [0]
for f in FOLDS:
    done = os.path.join(os.environ["nnUNet_results"],
                        f"Dataset801_IAC_LR/{TRAINER}__nnUNetPlans__3d_fullres/fold_{f}/checkpoint_final.pth")
    if os.path.isfile(done):
        print(f"fold {f} zaten bitmis, atlaniyor"); continue
    print(f"=== nnU-Net fold {f} egitiliyor ({TRAINER}) ===")
    # --c: varsa son checkpoint'ten devam et (Colab kopmalarina dayanikli)
    os.system(f"nnUNetv2_train 801 3d_fullres {f} -tr {TRAINER} --c")

# ADIM 4 — leakage-free Out-Of-Fold tahminler → `outputs/oof_probs`
Her fold, kendi validation vakalarını tahmin eder; birleştirince her development vakası
için 'eğitimde görülmemiş' bir tahmin çıkar. Flow'un prior'ı budur.

In [ ]:
!python nnunet/predict_oof.py --dataset 801 --config 3d_fullres \
    --trainer $TRAINER --splits configs/splits.json \
    --images "$IMAGES" --out outputs/oof_probs
print("oof npz:", len(os.listdir("outputs/oof_probs")))

# ADIM 5 — SDF cache'leri (flow'un x1 hedefi ve x0 başlangıcı)
`compute_gt_sdf` = GT'den milimetre SDF (x1 hedefi). `compute_coarse_sdf` = OOF tahminden
SDF (x0 başlangıç, leakage-free). Bir kez hesaplanır, her epoch yeniden hesaplanmaz.

In [ ]:
!python data/compute_gt_sdf.py --labels "$LABELS" --out outputs/gt_sdf --clip-mm 10
!python data/compute_coarse_sdf.py --oof outputs/oof_probs --ref-labels "$LABELS" \
    --out outputs/coarse_sdf --prob-thresh 0.5 --clip-mm 10
print("gt_sdf:", len(os.listdir("outputs/gt_sdf")), "| coarse_sdf:", len(os.listdir("outputs/coarse_sdf")))

---
# TRACK B — residual flow refinement

# ADIM 6 — flow eğitimi (fold 0)
Checkpoint'ler Drive'a yazılır (`--out`), `--resume` ile kopmadan sonra devam eder.
**Bu SAME hücreyi tekrar çalıştırınca** kaldığı yerden sürer. T4'te VRAM için patch=96
büyük gelirse `configs/flow.yaml` içinde `patch: 64` yap. Diğer fold'lar için `--fold 1..4`.

In [ ]:
FLOW_OUT = f"{DRIVE_RUNS}/flow_fold0"
!python flow/train.py --config configs/flow.yaml --fold 0 --resume \
    --images "$IMAGES" --labels "$LABELS" \
    --gt-sdf outputs/gt_sdf --coarse-sdf outputs/coarse_sdf \
    --out "$FLOW_OUT"
print("en iyi checkpoint ->", f"{FLOW_OUT}/best.pt")

# ADIM 7 (opsiyonel) — CV metrikleri
Bir tahmin klasörünü GT'ye karşı skorlar (Dice, HD95, clDice, NSD + topoloji). Önce bir
tahmin klasörü üretmen gerekir (nnU-Net baseline tahmini veya flow çıktısı). Örnek olarak
OOF tahminleri `.nii.gz` değil `.npz`; burada nnU-Net baseline tahmin klasörünü kullan.

```bash
# ornek: fold 0 val vakalarini baseline ile tahmin et, sonra skorla
# nnUNetv2_predict -i <val_imgs> -o outputs/preds_A -d 801 -c 3d_fullres -f 0 -tr $TRAINER --disable_tta
# python evaluation/evaluate_cv.py --pred outputs/preds_A --gt "$LABELS" --out outputs/cv_metrics.csv
```

## Notlar
- **Sıra önemli:** 1→2→(A2,A3)→4→5→6. OOF (adım 4) olmadan SDF cache (5) olmaz; onlar
  olmadan flow (6) olmaz. Bu yüzden eski notebook'ta flow hücreleri yorumdaydı.
- **En pahalı kısım nnU-Net 5-fold.** Bütçe azsa `FAST=True` ile tüm zinciri bir kez
  uçtan uca gör; sayılar zayıf ama boru hattının çalıştığını kanıtlar. Sonra `FAST=False`.
- **Her şey devam ettirilebilir:** nnU-Net `--c`, flow `--resume`, checkpoint'ler Drive'da.
- **External S seti** sadece en sonda, bir kez: `evaluation/evaluate_external.py`.